# Dialogue and Chatbots with Decoder-only Models (STUDENT version)

This notebook builds a mini-chatbot on top of a decoder-only Transformer (GPT/DialoGPT style).

**Learning goals**
- Load a decoder-only model.
- Implement a reply-generating function.
- Build a multi-turn dialogue with history.
- Explore decoding parameters.
- Use prompting for style/persona.
- Run a short qualitative evaluation.

## (Optional) Library installation / upgrade

If you run this in Google Colab, you may need to install or upgrade libraries. The command below is commented out by default; uncomment if needed.

In [ ]:
# TODO (optional): uncomment if you need to install/upgrade transformers in Colab
# !pip install -q "transformers>=4.40.0"

## Imports and basic setup

We import Hugging Face transformers, PyTorch, and optionally numpy for reproducibility helpers. Set the device (GPU if available) and optionally set seeds.

In [ ]:
# TODO: import the libraries
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch
# import numpy as np

# TODO: set the device
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)

# TODO (optional): set seeds for reproducibility
# torch.manual_seed(42)
# np.random.seed(42)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(42)

## Loading the decoder-only model and tokenizer

We will use `microsoft/DialoGPT-small` as the default model. It is trained for informal English dialogue, so outputs may be casual.

In [ ]:
# TODO: set the model name
model_name = "microsoft/DialoGPT-small"

# TODO: load tokenizer and model, and move model to device
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
# print(f"Loaded {model_name} on {device}")

## Single-turn generation

Generate a reply for a single prompt. Start with greedy decoding (deterministic) then try sampling for diversity. Suggested prompt: "Hello, how are you today?"

In [ ]:
# TODO: define a prompt to test
prompt = "Hello, how are you today?"

# TODO: tokenize the prompt (return_tensors="pt") and move to device
# inputs = tokenizer(prompt, return_tensors="pt").to(device)

# TODO: generate with greedy decoding first (do_sample=False)
# output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

# TODO: decode the full output and print
# generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
# print(f"Prompt: {prompt}\n\nModel reply (greedy): {generated_text}")

# TODO: try sampling-based decoding (e.g., do_sample=True, temperature=0.8, top_p=0.9)
# sampled_ids = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.9)
# sampled_text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
# print(f"Model reply (sampling): {sampled_text}")

## Representing the conversation history

Use a `history` list of strings like `"User: ..."` and `"Bot: ..."` (optionally a `"System: ..."` line). Implement `build_prompt(history, max_turns=4)` to keep only the last turns and join them with newlines.

In [ ]:
def build_prompt(history, max_turns=4):
    """
    history: list of strings like ["System: ...", "User: ...", "Bot: ..."]
    max_turns: keep only the last max_turns messages (None to keep all)
    """
    # TODO: select only the last max_turns messages (if max_turns is not None)
    # TODO: join them with newline characters to form the prompt text
    raise NotImplementedError("Implement build_prompt")

# TODO: create an example history and print the resulting prompt
history_example = [
    "System: You are a concise assistant.",
    "User: Hello!",
    "Bot: Hi there, how can I help?",
    "User: Tell me something fun.",
    "Bot: Here is a fun fact about space."
]
# TODO: print(build_prompt(history_example, max_turns=4))

## One-turn chat function (`chat_once`) using only newly generated tokens

Implement `chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs)` that:
1. Appends the new user message to history.
2. Builds a prompt via `build_prompt`.
3. Tokenizes and moves tensors to `device`.
4. Calls `model.generate(...)`.
5. Extracts only newly generated tokens (`generated_ids = output_ids[:, input_ids.shape[1]:]`).
6. Decodes to get `bot_reply` (fallback to placeholder if empty).
7. Appends the bot reply to history and returns `(history, bot_reply)`.

In [ ]:
def chat_once(history, user_utterance, max_new_tokens=60, **decoding_kwargs):
    """
    history: list of strings with "User:" / "Bot:" (and optional "System:")
    user_utterance: new user message to append
    """
    # TODO 1: append the new user message to history (with "User: ...")
    # TODO 2: build the prompt (optionally limit turns; None keeps all)
    # TODO 3: tokenize prompt and move tensors to device
    # TODO 4: call model.generate with max_new_tokens and decoding kwargs
    # TODO 5: slice to keep only newly generated tokens
    #   generated_ids = output_ids[:, input_ids.shape[1]:]
    # TODO 6: decode generated_ids to text (skip_special_tokens=True, strip)
    # TODO 7: if empty string, replace with placeholder "(I did not manage to generate a reply...)"
    # TODO 8: append bot reply to history with prefix "Bot: "
    # TODO 9: return history, bot_reply
    raise NotImplementedError("Implement chat_once")

# TODO: quick test call
# history = []
# history, reply = chat_once(history, "Hi! How are you?")
# print("Bot:", reply)

## Experiments with decoding parameters

Fix a user input such as `"Can you explain what a Transformer model is in simple terms?"`. Compare greedy decoding vs sampling (temperature/top-k/top-p) by running `chat_once` with different configurations.

In [ ]:
# TODO: set a base history (empty or with a system prompt)
base_history = []

user_input = "Can you explain what a Transformer model is in simple terms?"

# TODO: define decoding configurations to compare
decoding_setups = [
    # {"name": "greedy", "do_sample": False},
    # {"name": "temp0.7_top_p0.9", "do_sample": True, "temperature": 0.7, "top_p": 0.9},
    # {"name": "temp1.2_top_k50", "do_sample": True, "temperature": 1.2, "top_k": 50},
]

# TODO: loop through configs, reset history each time, call chat_once, and print replies
for cfg in decoding_setups:
    history = base_history.copy()
    history, bot_reply = chat_once(history, user_input, max_new_tokens=80, **cfg)
    print(f"=== Config: {cfg.get('name', 'unnamed')} ===")
    print("User:", user_input)
    print("Bot :", bot_reply)
    print()

## Controlling chatbot style with prompting

Use simple system prompts to steer tone, e.g.:
- `System: You are a polite and formal assistant. You answer concisely and respectfully.`
- `System: You are a friendly and slightly funny assistant. You sometimes make small jokes.`

Create separate histories for each style, run a few turns with `chat_once`, and compare the tone of answers.

In [ ]:
# TODO: define style system prompts
style_formal = "System: You are a polite and formal assistant. You answer concisely and respectfully."
style_funny = "System: You are a friendly and slightly funny assistant. You sometimes make small jokes."

# TODO: create histories for each style
history_formal = [style_formal]
history_funny = [style_funny]

# TODO: run a couple of turns for each style and compare
# Formal assistant
# history_formal, reply_f1 = chat_once(history_formal, "Could you summarize what transformers do?")
# print("Formal 1:", reply_f1)
# history_formal, reply_f2 = chat_once(history_formal, "And why are they useful?")
# print("Formal 2:", reply_f2)

# Funny assistant
# history_funny, reply_j1 = chat_once(history_funny, "Could you summarize what transformers do?")
# print("Funny 1:", reply_j1)
# history_funny, reply_j2 = chat_once(history_funny, "And why are they useful?")
# print("Funny 2:", reply_j2)

## Mini qualitative evaluation

Fill a small qualitative table for a few interactions. Use 1–5 scores (5 is best).

| Interaction summary | Coherence (1-5) | Relevance (1-5) | Fluency (1-5) | Tone/Politeness (1-5) | Notes |
| --- | --- | --- | --- | --- | --- |
| Example: Asked bot to explain transformers formally |  |  |  |  |  |
| Your test 1 |  |  |  |  |  |
| Your test 2 |  |  |  |  |  |

## Interactive chat loop (ChatGPT-style) – optional

Implement `chat_loop(system_prompt="System: You are a helpful assistant.", ...)` that:
- Initializes `history = [system_prompt]`.
- Prints an instruction line: type 'exit' or 'quit' to stop.
- In a `while True`, reads user input, breaks on exit words, otherwise calls `chat_once` and prints `Bot: ...`.
- Handles `EOFError` when no stdin is available.

In [ ]:
def chat_loop(system_prompt="System: You are a helpful assistant.", max_new_tokens=80, **decoding_kwargs):
    """
    Simple interactive loop; type 'exit' or 'quit' to stop.
    """
    # TODO: initialize history with the system prompt
    # TODO: print instructions
    # TODO: loop over user input with input("You: ")
    #   - handle exit/quit
    #   - call chat_once and print bot reply
    #   - handle EOFError if stdin is missing
    raise NotImplementedError("Implement chat_loop")

# TODO: uncomment to try the interactive loop (be mindful in non-interactive environments)
# chat_loop()

## Conclusions

Write 5–10 lines summarizing:
- Behaviour of decoder-only models in dialogue.
- Impact of decoding parameters.
- Effectiveness of simple prompting for style.
- Connection to course theory (Topic 4, sections 4.12–4.14).